Loading the data

In [15]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set up the visual style for charts
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

# Define the file path (adjust this if your path is different)
file_path = "data/raw_insurance_data.csv"

# Load the data
try:
    df = pd.read_csv(file_path)
    print("Data loaded successfully.")
except FileNotFoundError:
    print(f"Error: File not found at {file_path}. Please check your path.")
    # Exit or handle the error appropriately
    # If successful, proceed:
    print(f"\nTotal rows: {len(df)}")
    print("\nInitial Data Info:")
    df.info()

Error: File not found at data/raw_insurance_data.csv. Please check your path.

Total rows: 1000098

Initial Data Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000098 entries, 0 to 1000097
Data columns (total 52 columns):
 #   Column                    Non-Null Count    Dtype  
---  ------                    --------------    -----  
 0   UnderwrittenCoverID       1000098 non-null  int64  
 1   PolicyID                  1000098 non-null  int64  
 2   TransactionMonth          1000098 non-null  object 
 3   IsVATRegistered           1000098 non-null  bool   
 4   Citizenship               1000098 non-null  object 
 5   LegalType                 1000098 non-null  object 
 6   Title                     1000098 non-null  object 
 7   Language                  1000098 non-null  object 
 8   Bank                      854137 non-null   object 
 9   AccountType               959866 non-null   object 
 10  MaritalStatus             991839 non-null   object 
 11  Gender                

Memory Optimized data cleaning 

In [ ]:
import numpy as np
import pandas as pd

# Re-define the DataFrame if your kernel restarted (assuming 'df' is available)
# If your kernel didn't restart, ensure you run the date conversion first.
df['TransactionMonth'] = pd.to_datetime(df['TransactionMonth'])
print("TransactionMonth converted to datetime.")

# --- 2.2. Targeted Memory-Efficient Cleaning & Conversion ---

CLEANING_COLS = [
    'Citizenship', 'LegalType', 'Title', 'Language', 'Bank', 'AccountType', 
    'MaritalStatus', 'Gender', 'Country', 'Province', 'ItemType', 'VehicleType', 
    'bodytype', 'AlarmImmobiliser', 'TrackingDevice', 'CoverCategory', 
    'CoverType', 'CoverGroup', 'Section', 'Product'
]
placeholder_values = ['', ' ', 'Not specified', 'Unknown', '999', '9999', 'nan']

print(f"\nStarting targeted cleaning and 'category' conversion on {len(CLEANING_COLS)} columns.")

for col in CLEANING_COLS:
    # 1. Targeted replacement (only for the column, not the whole DataFrame)
    # The .isin() method is highly memory-efficient for replacement
    df.loc[df[col].isin(placeholder_values), col] = np.nan
    
    # 2. Immediate conversion to 'category'
    df[col] = df[col].astype('category')
    
print("Targeted columns cleaned and converted to 'category' dtype.")

# --- 2.3. Financial Data Sanity Check ---
# Ensure TotalPremium and TotalClaims are float and drop any resulting NaNs
df['TotalPremium'] = pd.to_numeric(df['TotalPremium'], errors='coerce')
df['TotalClaims'] = pd.to_numeric(df['TotalClaims'], errors='coerce')
df.dropna(subset=['TotalPremium', 'TotalClaims'], inplace=True)

print(f"\nData size after cleaning financial columns: {len(df)}")
print("\nMissing Value Counts for critical columns after cleaning:")
# Check NaNs after cleaning (Gender had placeholders like 'Not specified')
print(df[['Gender', 'Province', 'VehicleType']].isnull().sum())

# --- 2.4. Descriptive Statistics ---
print("\nDescriptive Statistics for Key Financial Variables:")
descriptive_stats = df[['TotalPremium', 'TotalClaims', 'CustomValueEstimate']].describe()
print(descriptive_stats)


# --- 2.5. Loss Ratio Calculation ---

def calculate_loss_ratio(group):
    """Calculates the Loss Ratio (TotalClaims / TotalPremium) for a given group."""
    total_claims = group['TotalClaims'].sum()
    total_premium = group['TotalPremium'].sum()

    if total_premium == 0:
        return np.nan
    return total_claims / total_premium

# 1. Overall Loss Ratio (LR)
overall_loss_ratio = calculate_loss_ratio(df)

print(f"\nOverall Portfolio Loss Ratio (TotalClaims / TotalPremium): {overall_loss_ratio:.4f}")

# 2. Segmented Loss Ratio
# Grouping operations are memory-efficient with 'category' data types.
lr_by_province = df.groupby('Province').apply(calculate_loss_ratio).dropna().sort_values(ascending=False)
print("\nLoss Ratio by Province (sorted highest to lowest):")
print(lr_by_province.to_string())

lr_by_vehicle = df.groupby('VehicleType').apply(calculate_loss_ratio).dropna().sort_values(ascending=False)
print("\nLoss Ratio by Vehicle Type (Top 5):")
print(lr_by_vehicle.head(5).to_string())

lr_by_gender = df.groupby('Gender').apply(calculate_loss_ratio).dropna().sort_values(ascending=False)
print("\nLoss Ratio by Gender:")
print(lr_by_gender.to_string())

Uncover low-risk segments & prepare for hypothesis testing and modeling

In [2]:
import pandas as pd
import numpy as np

# --- Configuration ---
# Set the correct file path (use the one that worked for you before: '../data/raw_insurance_data.csv')
file_path = '../data/raw_insurance_data.csv' 
CHUNK_SIZE = 50000  # Load 50,000 rows at a time
print(f"Attempting to load data in chunks of {CHUNK_SIZE} using separator '|'...")

# Critical categorical columns for cleaning and conversion
CLEANING_COLS = [
    'Citizenship', 'LegalType', 'Title', 'Language', 'Bank', 'AccountType', 
    'MaritalStatus', 'Gender', 'Country', 'Province', 'ItemType', 'VehicleType', 
    'bodytype', 'AlarmImmobiliser', 'TrackingDevice', 'CoverCategory', 
    'CoverType', 'CoverGroup', 'Section', 'Product'
]
placeholder_values = ['', ' ', 'Not specified', 'Unknown', '999', '9999', 'nan']


def process_chunk(chunk):
    """Applies cleaning and memory optimization to a single chunk of data."""
    
    # 1. Date Conversion
    chunk['TransactionMonth'] = pd.to_datetime(chunk['TransactionMonth'], errors='coerce')
    
    # 2. Targeted Cleaning & Conversion to 'category'
    for col in CLEANING_COLS:
        if col in chunk.columns:
            # Targeted replacement using .isin() for memory efficiency
            # Note: Categorical columns can still be stored as object when loaded in chunks, so this cleanup is vital
            chunk.loc[chunk[col].isin(placeholder_values), col] = np.nan
            
            # Immediate conversion to 'category'
            chunk[col] = chunk[col].astype('category')
            
    # 3. Financial Data Sanity Check (Coerce to numeric)
    chunk['TotalPremium'] = pd.to_numeric(chunk['TotalPremium'], errors='coerce')
    chunk['TotalClaims'] = pd.to_numeric(chunk['TotalClaims'], errors='coerce')
    
    # Drop rows with NaN financial data only (we're keeping category NaNs for analysis)
    chunk.dropna(subset=['TotalPremium', 'TotalClaims'], inplace=True)
    
    return chunk


# --- Chunking Execution ---
df_list = []
try:
    # --- FIX APPLIED HERE: sep='|' ---
    chunk_iterator = pd.read_csv(file_path, sep='|', chunksize=CHUNK_SIZE, low_memory=False)
    
    # Process each chunk
    for i, chunk in enumerate(chunk_iterator):
        processed_chunk = process_chunk(chunk)
        df_list.append(processed_chunk)
        if (i + 1) % 5 == 0:
            print(f"Processed {i + 1} chunks (approx. {len(df_list) * CHUNK_SIZE / 1000000:.1f} million records)...")

    # Concatenate all chunks back into the final DataFrame
    df = pd.concat(df_list, ignore_index=True)
    print("\nData loading and cleaning COMPLETE.")

except Exception as e:
    print(f"\nFATAL ERROR during data loading: {e}")
    # Stop execution if loading fails
    raise


# --- 2.3. Final Statistics and Loss Ratio Calculation ---

print(f"\nFinal cleaned Data Shape: {df.shape}")
print("\nMissing Value Counts for critical columns after cleaning:")
print(df[['Gender', 'Province', 'VehicleType']].isnull().sum())

print("\nDescriptive Statistics for Key Financial Variables:")
descriptive_stats = df[['TotalPremium', 'TotalClaims', 'CustomValueEstimate']].describe()
print(descriptive_stats)


def calculate_loss_ratio(group):
    """Calculates the Loss Ratio (TotalClaims / TotalPremium) for a given group)."""
    total_claims = group['TotalClaims'].sum()
    total_premium = group['TotalPremium'].sum()

    # If the group has zero premium (very rare, but possible), return NaN
    if total_premium == 0:
        return np.nan
    return total_claims / total_premium

# 1. Overall Loss Ratio (LR)
overall_loss_ratio = calculate_loss_ratio(df)
print(f"\nOverall Portfolio Loss Ratio (TotalClaims / TotalPremium): {overall_loss_ratio:.4f}")

# 2. Segmented Loss Ratio
lr_by_province = df.groupby('Province').apply(calculate_loss_ratio).dropna().sort_values(ascending=False)
print("\nLoss Ratio by Province (sorted highest to lowest):")
print(lr_by_province.to_string())

lr_by_vehicle = df.groupby('VehicleType').apply(calculate_loss_ratio).dropna().sort_values(ascending=False)
print("\nLoss Ratio by Vehicle Type (Top 5):")
print(lr_by_vehicle.head(5).to_string())

lr_by_gender = df.groupby('Gender').apply(calculate_loss_ratio).dropna().sort_values(ascending=False)
print("\nLoss Ratio by Gender:")
print(lr_by_gender.to_string())

Attempting to load data in chunks of 50000 using separator '|'...
Processed 5 chunks (approx. 0.2 million records)...
Processed 10 chunks (approx. 0.5 million records)...
Processed 15 chunks (approx. 0.8 million records)...
Processed 20 chunks (approx. 1.0 million records)...

Data loading and cleaning COMPLETE.

Final cleaned Data Shape: (1000098, 52)

Missing Value Counts for critical columns after cleaning:
Gender         950526
Province            0
VehicleType       552
dtype: int64

Descriptive Statistics for Key Financial Variables:
       TotalPremium   TotalClaims  CustomValueEstimate
count  1.000098e+06  1.000098e+06         2.204560e+05
mean   6.190550e+01  6.486119e+01         2.255311e+05
std    2.302845e+02  2.384075e+03         5.645157e+05
min   -7.825768e+02 -1.200241e+04         2.000000e+04
25%    0.000000e+00  0.000000e+00         1.350000e+05
50%    2.178333e+00  0.000000e+00         2.200000e+05
75%    2.192982e+01  0.000000e+00         2.800000e+05
max    6.52826

C:\Users\bezaw\AppData\Local\Temp\ipykernel_13640\285751294.py:95: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  lr_by_province = df.groupby('Province').apply(calculate_loss_ratio).dropna().sort_values(ascending=False)



Loss Ratio by Province (sorted highest to lowest):
Province
Gauteng          1.222018
KwaZulu-Natal    1.082693
Western Cape     1.059472
North West       0.790367
Mpumalanga       0.720897
Free State       0.680758
Limpopo          0.661199
Eastern Cape     0.633813
Northern Cape    0.282699


C:\Users\bezaw\AppData\Local\Temp\ipykernel_13640\285751294.py:99: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  lr_by_vehicle = df.groupby('VehicleType').apply(calculate_loss_ratio).dropna().sort_values(ascending=False)



Loss Ratio by Vehicle Type (Top 5):
VehicleType
Heavy Commercial     1.628112
Medium Commercial    1.050251
Passenger Vehicle    1.048198
Light Commercial     0.232066
Bus                  0.137292

Loss Ratio by Gender:
Gender
Male      0.883910
Female    0.821879


C:\Users\bezaw\AppData\Local\Temp\ipykernel_13640\285751294.py:103: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  lr_by_gender = df.groupby('Gender').apply(calculate_loss_ratio).dropna().sort_values(ascending=False)


Loading data in chunks...


FileNotFoundError: [Errno 2] No such file or directory: '..\\data\\raw\\raw_insurance_data.csv'